# Research-only official wheelhouse byte audit
No training or evaluation. Reads mounted wheels as ZIP files.

In [ ]:
import hashlib,json,zipfile,datetime
from pathlib import Path
root=Path('/kaggle/input')
out=Path('/kaggle/working/research_wheelhouse_audit');out.mkdir()
prefixes=('swegemma-','adk_submission-','google_adk-','google_genai-','vllm-','transformers-','compressed_tensors-')
report={'at_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'mode':'CPU static wheel inspection; no install, import, model, adapter, training or inference','wheels':{},'sources':{}}
terms=('thinking_budget','seed','reasoning_content','max_lora_rank','max_loras','discover_adapters','vision_embedder','zero_','json.dumps','token_threshold','32768','16384')
for p in sorted(root.rglob('*.whl')):
 if not p.name.startswith(prefixes):continue
 h=hashlib.sha256()
 with p.open('rb') as f:
  for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
 report['wheels'][p.name]={'sha256':h.hexdigest(),'bytes':p.stat().st_size,'mount':str(p)}
 with zipfile.ZipFile(p) as z:
  for n in z.namelist():
   if not n.endswith('.py'):continue
   keep=(n.startswith(('swegemma/','adk_submission/')) or n in ('google/adk/models/lite_llm.py','vllm/model_executor/models/gemma4.py','vllm/entrypoints/chat_utils.py'))
   if not keep:continue
   b=z.read(n);s=b.decode('utf-8',errors='replace');lines=s.splitlines()
   hits=[{'line':i+1,'text':line.strip()[:350]} for i,line in enumerate(lines) if any(t in line for t in terms)]
   report['sources'][n]={'sha256':hashlib.sha256(b).hexdigest(),'bytes':len(b),'hits':hits}
   target=out/'source'/n;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(b)
(out/'audit.json').write_text(json.dumps(report,indent=2))
print(json.dumps({'wheels':report['wheels'],'source_count':len(report['sources'])},indent=2))
